In [6]:
import helper_functions
from sklearn.model_selection import train_test_split
import pandas as pd
from xgboost import XGBRegressor
import xgboost as xgb
import tensorflow as tf
mape = tf.keras.losses.MeanAbsolutePercentageError()
mse = tf.keras.losses.MeanSquaredError()
mae = tf.keras.losses.MeanAbsoluteError()
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_absolute_percentage_error
import numpy as np
import sklearn

In [2]:
data_df = helper_functions.get_data()
data_df = helper_functions.drop_unmatched_rows(data_df)
data_df = data_df.rename(columns={'machine': 'source machine'})
data_df = helper_functions.create_machine_combinations(data_df)
data_df = helper_functions.merge_benchmarks(data_df, False)
data_df = helper_functions.calc_relative_performances(data_df)


refined_df = helper_functions.remove_unneeded_columns(data_df)
run_data, relative_performance_values = helper_functions.split_x_y(refined_df)

x_all, y_all = run_data, relative_performance_values
x_train, x_test, y_train, y_test = train_test_split(run_data, relative_performance_values, test_size = 0.20, random_state = 0)

/home/alex/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:117: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raja_df.machine[raja_df['machine'] == 'ec2-c5n'] = 'ec2-c5.metal'
/home/alex/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raja_df.machine[raja_df['machine'] == 'ec2-c6i'] = 'ec2-c6i.metal'


In [3]:
# n_estimators, min_mape = helper_functions.tune_n_estimators(x_all, y_all, 1450, 1500)
# max_depth, random_split_mae = helper_functions.tune_depth(x_all, y_all, n_estimators)
n_estimators = 1500
max_depth = 6

In [5]:
model = XGBRegressor(n_estimators=n_estimators, max_depth=max_depth)
model.fit(x_train, y_train, eval_metric='mae')
def calc_mae(y_true, y_pred):
    return np.mean(np.absolute(y_true - y_pred))
def calc_mape(y_true, y_pred):
    return np.mean(np.absolute((y_true - y_pred) / (y_true)))
y_pred = model.predict(x_test)
print('mae', calc_mae(y_test, y_pred))
print('mape', calc_mape(y_test, y_pred))
print(np.mean(y_pred), np.mean(y_test))
print(np.std(y_pred), np.std(y_test))
len(y_test[y_test == 0])

/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  elif is_categorical_dtype(dtype) and enable_categorical:
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:332: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if is_categorical_dtype(dtype)
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:323: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future versio

mae 2.58189078552858
mape 0.10056030538476873
770.74994 773.272808450381
11967.143 11968.501900457062


/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  elif is_categorical_dtype(dtype) and enable_categorical:
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:332: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if is_categorical_dtype(dtype)
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:323: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future versio

0

In [8]:
# mape_scorer = make_scorer(helper_functions.MAPE)
# mape_scorer = helper_functions.get_mape_scorer()
mape_scorer = make_scorer(sklearn.metrics.mean_absolute_percentage_error)
model = XGBRegressor(n_estimators=n_estimators, max_depth=max_depth)
# define model evaluation method
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=1)
# evaluate model
scores = cross_val_score(model, x_all, y_all, scoring=mape_scorer, cv=cv, n_jobs=-1)

# force scores to be positive
scores = np.absolute(scores)
print('Mean MAPE: %.3f (%.3f)' % (scores.mean(), scores.std()) )

random_mapes = scores

model = XGBRegressor(n_estimators=n_estimators, max_depth=max_depth)
scores = cross_val_score(model, x_all, y_all, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
scores = np.absolute(scores)

random_maes = scores

/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  elif is_categorical_dtype(dtype) and enable_categorical:
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(

Mean MAPE: 0.316 (0.723)


/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:427: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(data):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:427: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(data):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  elif is_cat